In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# 1. Generate a synthetic binary classification dataset
X, y = make_classification(
    n_samples=2000, 
    n_features=20, 
    n_informative=15, 
    random_state=42
)

# Split into train, calibration, and test subsets
# Note: Prefit requires disjoint datasets for training and calibration
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.5, random_state=42)
X_calib, X_test, y_calib, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


# =====================================================================
# APPROACH 1: Calibration using Integrated Cross-Validation (cv=5)
# =====================================================================
# The base estimator is cloned and fit on K-1 folds, then calibrated on the remaining fold.
# This approach maximizes data efficiency when your training set is small.

base_svc_cv = LinearSVC(random_state=42, dual="auto")
calibrated_cv = CalibratedClassifierCV(estimator=base_svc_cv, method='sigmoid', cv=5)

# Combines training and calibration in a single step
calibrated_cv.fit(X_train, y_train)

# Predict probabilities on test data
probs_cv = calibrated_cv.predict_proba(X_test)[:, 1]
brier_cv = brier_score_loss(y_test, probs_cv)


# =====================================================================
# APPROACH 2: Calibration using an Already Fitted Model (cv='prefit')
# =====================================================================
# Use this when your base model takes a long time to train, or you want to keep
# your primary training and calibration processes completely isolated.

# Step A: Fit the base model on the training set
prefit_svc = LinearSVC(random_state=42, dual="auto")
prefit_svc.fit(X_train, y_train)

# Step B: Wrap the fit model and pass cv='prefit'
calibrated_prefit = CalibratedClassifierCV(estimator=prefit_svc, cv='prefit', method='sigmoid')

# Step C: Calibrate parameters strictly using the separate calibration set
calibrated_prefit.fit(X_calib, y_calib)

# Predict probabilities on test data
probs_prefit = calibrated_prefit.predict_proba(X_test)[:, 1]
brier_prefit = brier_score_loss(y_test, probs_prefit)


# =====================================================================
# Display Results
# =====================================================================
print(f"Brier Score Loss (Cross-Validation Calibration): {brier_cv:.4f}")
print(f"Brier Score Loss (Prefit Calibration Layout):   {brier_prefit:.4f}")
# (Lower Brier Score indicates superior probability calibration closer to actual outcomes)


InvalidParameterError: The 'cv' parameter of CalibratedClassifierCV must be an int in the range [2, inf), an object implementing 'split' and 'get_n_splits', an iterable or None. Got 'prefit' instead.